<a href="https://colab.research.google.com/github/SuryamaheshPenke/Gen_AI-Notes/blob/main/Class_6_Few_Shot%2C_Zero_Shot_and_Prompt_Optimisation_(Advanced).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Class 6 - Few-Shot, Zero-Shot and Prompt Optimisation (Advanced)**
### Module 1: Prompt Engineering & LLM Mastery

---

> **This class builds directly on Class 5 (Prompt Design Patterns).** In Class 5, you learned the 4-part prompt structure and the difference between zero-shot and few-shot prompting. This class goes deeper: why zero-shot chain-of-thought actually works, how to design few-shot examples that consistently improve outputs, how to retrieve the right examples dynamically, and when the whole prompt-engineering game should be handed off to an automated optimizer.

## **Agenda**

- **PART 1 - Zero-Shot Chain-of-Thought: Why It Works**
  - The mechanism behind "think step by step"
  - What the model is actually doing differently
- **PART 2 - Designing Few-Shot Examples**
  - Selection criteria: what makes a good example
  - Ordering effects: sequence changes output quality
  - Diversity requirements: coverage over repetition
- **PART 3 - Dynamic Few-Shot Retrieval**
  - The problem with static example sets
  - Embedding-based retrieval: pick the best examples per query
- **PART 4 - DSPy: Compiling Prompts Instead of Writing Them**
  - What DSPy is and why it exists
  - The mental model: prompts as learnable programs
  - What the API looks like (conceptual)
- **PART 5 - Decision Framework: Prompt vs Automate**
  - When hand-crafted prompts are the right call
  - When to reach for optimization tools
- **Code Demos**
- **Key Takeaways**
- **Quick Check**

## **PART 1 - Zero-Shot Chain-of-Thought: Why It Works**

### Context

In Class 5, you saw that adding "let's think step by step" to a prompt improves model accuracy on reasoning tasks. That was stated as a fact. This part explains the mechanism - why a four-word phrase changes what the model produces.

The discovery came from Kojima et al. (2022), "Large Language Models are Zero-Shot Reasoners." They found that this phrase alone, added to a prompt with no examples, pushed GPT-3 accuracy on math word problems from 17.7% to 78.7%. That is not a small improvement. Understanding why it happens changes how you use it.

---

**Simple**

When you ask a model a question directly, it reaches for the most statistically likely answer - the shortcut. When you ask it to "think step by step," you are forcing the model to generate intermediate tokens before it reaches the answer. Those intermediate tokens - the reasoning steps - act as working memory. The model conditions its next token on all the previous tokens, including the reasoning it just wrote. Better reasoning steps lead to a better final token.

Think of it like showing your work in math class. Writing out the steps does not just help the teacher check your answer - it actually helps you avoid arithmetic mistakes because each line builds on the last.

**Technical**

Transformers generate text autoregressively: each output token is sampled from a probability distribution conditioned on all preceding tokens. When a model generates a direct answer, it is conditioning only on the input prompt tokens. When you instruct it to reason first, the model generates a chain of reasoning tokens $r_1, r_2, ..., r_k$ before generating the answer token $a$. The answer is now conditioned on $P(a | prompt, r_1, ..., r_k)$ rather than $P(a | prompt)$. The intermediate reasoning tokens function as soft compute - they expand the effective depth of computation beyond the fixed transformer layers. This is why zero-shot CoT degrades on tasks that require no multi-step reasoning: the extra tokens add noise, not signal.

## **PART 2 - Designing Few-Shot Examples**

### Context

Class 5 introduced few-shot prompting as "showing the model examples." That framing is correct but incomplete. Not all examples are equal. A bad few-shot set can hurt performance relative to zero-shot. A carefully designed set can push accuracy past what fine-tuning achieves on small datasets.

Three variables drive few-shot quality: which examples you pick, in what order you present them, and how much they cover the space of possible inputs.

---

### Selection Criteria

**Simple**

The best examples are similar to the input the model will see, but not identical to each other. You want examples that are close to the problem at hand (so the model learns the right pattern) but varied enough that the model does not just copy a template. Think of it like a study guide: five versions of the same problem type teach you nothing new after the first one.

**Technical**

Example selection should optimize for two properties simultaneously: relevance and diversity. Relevance is typically measured by semantic similarity between the example input and the target query - embeddings from a sentence transformer model work well here. Diversity is measured by coverage of the input space, often approximated by clustering examples and selecting one per cluster. Studies (Liu et al., 2021 "What Makes Good In-Context Examples for GPT-3?") show that random selection underperforms similarity-based selection, and similarity-only selection underperforms similarity + diversity selection.

---

### Ordering Effects

**Simple**

The order of your examples matters more than you might expect. Models tend to weight recent examples more heavily. If your last example shows a conservative, cautious output, the model will lean conservative. If it shows a creative, expansive output, the model leans that way. The last example acts as a strong prior on format and style.

**Technical**

This phenomenon is partially explained by the recency bias in attention: later tokens in the context window attend more strongly to recent tokens due to position encoding effects. Lu et al. (2021) "Fantastically Ordered Prompts" demonstrated variance of up to 35% in task accuracy across different orderings of the same few-shot examples. The recommendation from the literature is to place the most prototypical (most representative of the average case) example last, ensuring the final prior is well-calibrated rather than skewed by an edge case.

---

### Diversity Requirements

**Simple**

If all your examples are from one category or style, the model learns to output that category even when the input is from a different one. You want your example set to cover the range of inputs the model will actually encounter. Three diverse examples often outperform ten similar ones.

**Technical**

Diversity in few-shot sets reduces label bias - the tendency for models to output the label distribution they see in the examples regardless of the actual input. If 4 out of 5 examples have a "positive" label, the model biases toward positive outputs. Calibrated diversity also prevents the model from latching onto surface features (e.g., all positive examples happen to use formal language) rather than the intended semantic signal. The minimum recommended diversity threshold is coverage across all expected output classes, with additional diversity along input surface-form axes (length, vocabulary, domain).

## **PART 3 - Dynamic Few-Shot Retrieval**

### Context

Static few-shot sets - where you hardcode the same 3-5 examples into every prompt - are the most common approach. They are also the most brittle. A fixed set of examples optimized for one query distribution will degrade when the input shifts. Dynamic retrieval solves this: instead of one example set for all queries, you maintain a library of examples and retrieve the best subset for each specific query.

---

**Simple**

Imagine you are a tutor with a drawer full of practice problems. Instead of giving every student the same five problems, you look at their specific question first and pull out the five problems that are most similar to what they are struggling with. Dynamic few-shot retrieval does this automatically using embeddings: the query is converted to a vector, example inputs are converted to vectors, and the closest matches are pulled at runtime.

**Technical**

Dynamic few-shot retrieval uses a two-stage architecture. First, an offline indexing step: all candidate examples are embedded using a text embedding model (e.g., `text-embedding-3-small` from OpenAI, or a sentence-transformers model) and stored in a vector database (vector databases are covered in Class 23 - for now, think of it as a searchable store of embeddings) or an in-memory index. Second, an online retrieval step: at inference time, the incoming query is embedded with the same model, and cosine similarity is computed against the stored embeddings. The top-k most similar examples are selected and inserted into the prompt. This approach has two constraints: embedding model must be consistent between index and query time, and the retrieved examples plus the query must fit within the context window. The retrieval adds latency (one embedding call + nearest-neighbor search), which is typically 20-100ms and acceptable for most applications.

## **PART 4 - DSPy: Compiling Prompts Instead of Writing Them**

> **Setup note:** DSPy requires installation: `pip install dspy-ai openai`. Hands-on use starts in Module 2 once Python fundamentals and LLM SDK basics are in place. This section introduces the concept so you understand *why* it exists before you build with it.

### What DSPy Is

DSPy (Declarative Self-improving Python, Khattab et al. 2022) reframes prompting as program compilation. Instead of writing "You are a helpful assistant, here are three examples, now do X," you write a Python class that describes the task signature - what inputs go in, what outputs come out. DSPy then generates the prompt text. If you give it examples and a scoring metric, it can optimize the prompt automatically by trying variations and keeping what works. Think of the difference between writing a sort function by hand versus calling `sort()` - the task still happens, you just stopped managing the internals.

### When to Use DSPy

DSPy requires two things you often do not have when starting a new task: a labeled dataset (50+ examples minimum) and a scalar metric function (accuracy, F1, or a custom evaluator). Without both, you cannot run an optimizer. Use DSPy when: the task is complex enough that no single hand-crafted prompt handles the full input distribution, you have collected failures and annotated them, and you need the prompt to stay optimized as inputs drift over time. For everything else - new tasks, fast experiments, tasks without measurable ground truth - write prompts manually first.

### What the API Looks Like

Here is a sketch of DSPy code to make the abstraction concrete. This is not for running today - it is to show you what "treating prompts as learnable parameters" looks like in practice:

```python
import dspy

lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm)

# Define the task signature: what goes in, what comes out
class ClassifyIntent(dspy.Signature):
    """Classify the intent of a customer support message."""
    message: str = dspy.InputField()
    intent: str = dspy.OutputField(desc="one of: billing, technical, general")

# Define the module - DSPy handles the prompt internally
classifier = dspy.Predict(ClassifyIntent)
result = classifier(message="My invoice shows the wrong amount")
print(result.intent)  # billing

# To optimize: give it examples and a metric, then compile
trainset = [dspy.Example(message="...", intent="billing").with_inputs("message")]
optimizer = dspy.BootstrapFewShot(metric=lambda ex, pred, **_: ex.intent == pred.intent)
optimized = optimizer.compile(classifier, trainset=trainset)
```

The prompt you never write gets generated and optimized automatically. The code describes *what* the task does, not *how* to phrase it. **Hands-on DSPy is in Module 2.**

## **PART 5 - Decision Framework: When to Prompt vs When to Automate**

### Context

Having DSPy as an option does not mean every prompt should go through an optimizer. Manual prompt engineering is faster to start, easier to debug, and sufficient for many production use cases. Knowing when to switch is a practical engineering decision.

---

**Simple**

Use hand-crafted prompts when you can look at a few outputs and tell whether they are right or wrong, when the task is stable and well-defined, and when you are moving fast. Use automated optimization when the task is complex enough that no one prompt works consistently, when you have a dataset and a clear metric, and when the prompt needs to stay optimized as inputs drift over time.

The rule of thumb: if you have spent more than two hours tuning a prompt by hand and it still fails on 20% of inputs, that is a signal to reach for optimization.

**Technical**

The decision hinges on three conditions:

1. **Metric availability** - Automated optimization requires a scalar metric (accuracy, F1, BLEU, custom function). If you cannot define what "correct" means programmatically, you cannot run an optimizer.

2. **Dataset availability** - You need at least 20-50 labeled examples for BootstrapFewShot, more for MIPRO. Without labeled data, manual prompting is the only option.

3. **Task complexity** - Single-turn, single-step tasks with clear formatting requirements are well-served by manual prompts. Multi-hop reasoning chains, retrieval-augmented generation pipelines, and agent loops have too many interacting prompt surfaces for manual tuning to be reliable at scale.

A practical heuristic: prompt manually first, collect failures, annotate them, then run an optimizer once you have 50+ labeled examples. Use the optimizer's output as a new baseline, then tune manually from there if needed.

| Condition | Manual Prompting | Automated Optimization |
|---|---|---|
| Starting a new task | Yes | No |
| Have labeled data (50+) | Optional | Yes |
| Complex multi-step pipeline | Hard | Yes |
| Need to explain the prompt | Yes | Harder |
| Task inputs shift over time | Re-tune manually | Re-compile |

# **Code Demos**

> Run each cell independently. Each demo shows exactly one concept. Read the output notes after each cell.

In [ ]:
# Setup - run this first
!pip install openai scikit-learn numpy --quiet

import os
from openai import OpenAI

# Set your API key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # replace with your key
client = OpenAI()

print("Setup complete.")

### **Demo 1 - Zero-Shot vs Zero-Shot CoT**

Same question, same model, same temperature. Only difference: one asks directly, the other adds "think step by step."

In [ ]:
def ask(prompt, label):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    print(f"--- {label} ---")
    print(response.choices[0].message.content)
    print()

problem = """
A farmer has 17 sheep. All but 9 die. How many sheep are left?
"""

# Zero-shot: direct question
ask(problem, "Zero-Shot (Direct)")

# Zero-shot CoT: same question, one added instruction
ask(problem + "\nThink step by step.", "Zero-Shot CoT")

Look at the intermediate reasoning the CoT version produces before reaching the answer. The phrase "all but 9" is a language trap - the direct version often misreads it. The CoT version unpacks the wording first, which changes the answer token it produces. Both prompts are zero-shot - no examples provided.

### **Demo 2 - Few-Shot Ordering Effects**

Same three examples, two different orderings. The last example acts as the strongest prior on output format.

In [ ]:
# Sentiment classification with few-shot
# Three examples: positive, negative, neutral
# We test two orderings and see how the output changes

query = "The product works but I expected more for the price."

# Ordering A: neutral last
examples_neutral_last = """
Review: "This laptop is the best I've ever owned."
Sentiment: positive

Review: "Completely broke after one week. Waste of money."
Sentiment: negative

Review: "It's okay. Does what it says on the box."
Sentiment: neutral
"""

# Ordering B: negative last
examples_negative_last = """
Review: "This laptop is the best I've ever owned."
Sentiment: positive

Review: "It's okay. Does what it says on the box."
Sentiment: neutral

Review: "Completely broke after one week. Waste of money."
Sentiment: negative
"""

for ordering_name, examples in [("Neutral Last", examples_neutral_last), ("Negative Last", examples_negative_last)]:
    prompt = f"{examples}\nReview: \"{query}\"\nSentiment:"
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=10
    )
    result = response.choices[0].message.content.strip()
    print(f"Ordering: {ordering_name} -> Predicted: {result}")

The query is genuinely ambiguous - it could be neutral or negative. Watch how the predicted label shifts depending on which example appears last. This is ordering bias in practice. For production systems, the recommendation is to place the most representative, unambiguous example last.

### **Demo 3 - Dynamic Few-Shot Retrieval with Embeddings**

A small example library. For each incoming query, retrieve the top-2 most similar examples by cosine similarity, then use them as the few-shot context.

In [ ]:
# Note: This demo uses embeddings covered in Module 3 (Class 22). Run to see output; full understanding comes after that class.

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Example library: input-output pairs
example_library = [
    {"input": "My payment failed but I was still charged.", "output": "billing"},
    {"input": "I cannot log into my account.", "output": "technical"},
    {"input": "Where can I find my invoice?", "output": "billing"},
    {"input": "The app crashes when I open it.", "output": "technical"},
    {"input": "How do I cancel my subscription?", "output": "billing"},
    {"input": "I want to speak to a human agent.", "output": "general"},
    {"input": "My password reset email never arrived.", "output": "technical"},
    {"input": "Do you offer student discounts?", "output": "general"},
]

def get_embeddings(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return np.array([e.embedding for e in response.data])

# Embed all examples once (offline indexing step)
example_inputs = [e["input"] for e in example_library]
example_embeddings = get_embeddings(example_inputs)
print(f"Indexed {len(example_library)} examples.")

def retrieve_examples(query, k=2):
    query_embedding = get_embeddings([query])
    similarities = cosine_similarity(query_embedding, example_embeddings)[0]
    top_k_indices = np.argsort(similarities)[::-1][:k]
    return [example_library[i] for i in top_k_indices], similarities[top_k_indices]

def classify_with_dynamic_few_shot(query):
    retrieved, scores = retrieve_examples(query, k=2)

    # Build few-shot prompt from retrieved examples
    few_shot_text = ""
    for ex in retrieved:
        few_shot_text += f'Message: "{ex["input"]}"\nIntent: {ex["output"]}\n\n'

    prompt = few_shot_text + f'Message: "{query}"\nIntent:'

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=10
    )
    prediction = response.choices[0].message.content.strip()

    print(f"Query: {query}")
    print(f"Retrieved examples:")
    for ex, score in zip(retrieved, scores):
        print(f"  [{score:.3f}] {ex['input']} -> {ex['output']}")
    print(f"Prediction: {prediction}")
    print()

# Test with different query types
classify_with_dynamic_few_shot("The billing page shows an error.")
classify_with_dynamic_few_shot("What are your office hours?")

Watch the similarity scores and which examples get retrieved for each query. A billing-related query pulls billing examples; a general inquiry pulls general ones. The prompt changes per query - same library, different context each time. In a production system, this library can have thousands of examples and the retrieval still runs in under 100ms.

### **Demo 4 - Few-Shot Quality: Diverse vs Repetitive Examples**

Same number of examples (3), same task. One set uses diverse examples, one uses repetitive examples from the same sub-type.

In [ ]:
# Task: extract the action item from a meeting note

query_note = "Priya mentioned that the onboarding flow needs a loading state before the dashboard renders."

# Repetitive examples: all about UI bugs
repetitive_examples = """
Note: "The button color is wrong on the settings page."
Action item: Fix button color on settings page.

Note: "The modal closes too slowly on mobile."
Action item: Improve modal close animation speed on mobile.

Note: "The font size is too small in the footer."
Action item: Increase footer font size.
"""

# Diverse examples: different domains, different types of action items
diverse_examples = """
Note: "We need to migrate the old user data before the next release."
Action item: Complete user data migration before next release.

Note: "The API response time is above 2 seconds for the search endpoint."
Action item: Optimize search endpoint to reduce response time below 2 seconds.

Note: "Sales wants a weekly report on new signups sent every Monday."
Action item: Set up automated weekly signup report for Sales, delivered Mondays.
"""

for set_name, examples in [("Repetitive (all UI bugs)", repetitive_examples), ("Diverse (varied domains)", diverse_examples)]:
    prompt = f"{examples}\nNote: \"{query_note}\"\nAction item:"
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    result = response.choices[0].message.content.strip()
    print(f"--- {set_name} ---")
    print(result)
    print()

The repetitive example set teaches the model to produce short, UI-specific action items. The diverse set teaches the model the actual pattern - extract the task, make it actionable, include relevant context. The query is about a front-end feature but the expected output is a full action item, not a bug fix note. Compare which output is actually usable.

# **Key Takeaways**

1. Zero-shot CoT works because intermediate reasoning tokens change the conditional probability of the answer token - the model conditions on its own reasoning, not just the input. It degrades on tasks that require no multi-step inference.

2. Few-shot example selection should optimize for both relevance (similar to the query) and diversity (covering the input space). Relevance-only selection outperforms random, but relevance + diversity outperforms both.

3. Example ordering is not cosmetic. The last example exerts the strongest influence on output format and style. Place the most representative example last.

4. Dynamic few-shot retrieval uses embedding similarity to select the best examples per query at runtime. This removes the need to hand-pick a static example set and scales to large example libraries without increasing prompt size.

5. DSPy reframes prompting as program compilation. You define the task signature (inputs and outputs), DSPy generates and optimizes the prompt text. It requires labeled data and a metric - without both, you cannot run an optimizer.

6. Manual prompting is the right default. Switch to automated optimization when you have 50+ labeled examples, a defined metric, and a task complex enough that no single hand-crafted prompt handles the full input distribution reliably.

7. These techniques compose. A production pipeline might use dynamic retrieval to populate few-shot slots, CoT for the reasoning step, and DSPy to optimize the instruction text - each layer solving a different part of the reliability problem.

# **Quick Check**

1. A model answers a math word problem incorrectly with zero-shot prompting but correctly with zero-shot CoT. What is the mechanical explanation for why the CoT version produces a better answer?

2. You have a few-shot set for classifying customer support tickets into 5 categories. All 5 examples happen to use formal, polite language. The model starts classifying casual-language tickets incorrectly. What dimension of diversity is missing from your example set?

3. You are building a few-shot prompt for a task and have 20 candidate examples. How would you decide which 5 to include? What would you optimize for, and what tool would you use to measure it?

4. You test two orderings of the same 4 few-shot examples and get different model outputs on an ambiguous input. Which position in the sequence has the strongest influence on the output, and why?

5. DSPy's `BootstrapFewShot` optimizer generates few-shot examples automatically. What two things do you need to provide before you can run it? What happens if you do not have them?

6. You are starting a new feature that requires an LLM to classify text into 3 categories. You have no labeled data yet. Should you use DSPy or write a prompt manually? Walk through the decision using the framework from Part 5.

7. A teammate says: "Dynamic few-shot retrieval is overkill - just use the same 5 examples every time." Under what conditions would they be right? Under what conditions would they be wrong?

## **Quick Check - Answers**

---

*Attempt the questions above before reading these answers.*

---

**1.** Zero-shot CoT causes the model to generate intermediate reasoning tokens before the final answer. Each step becomes part of the context, and the model attends to its own intermediate reasoning when generating the next step. The final answer token generation sees a richer context than the question alone - the model can "use" its earlier computations.

**2.** Style diversity is missing. All 5 examples use formal language, so the model learned category boundaries only for formal text. Add examples that cover casual, informal, and fragmented writing across the same 5 categories so the model learns to classify independent of writing style.

**3.** Optimize for diversity and coverage: select examples that together span all output categories, varied input formats, and genuinely ambiguous cases. Use a held-out test set to score accuracy before and after swapping each candidate in. Prefer examples where the correct label is non-obvious - those carry the most discriminative signal.

**4.** The last example in the sequence has the strongest influence due to recency bias in attention - the model's attention weights are higher for more recent tokens. The final example functions as an implicit template for the output immediately following it.

**5.** You need: (a) a training set of labeled input-output pairs, and (b) a metric function that scores model outputs against correct answers. Without labeled data, BootstrapFewShot cannot generate candidate examples; without a metric, it cannot evaluate which examples improve performance.

**6.** Write the prompt manually first. DSPy requires labeled data and a metric function - neither exists yet. Start with zero-shot or a hand-crafted 3-example few-shot. Switch to DSPy once you have 50+ labeled examples and a clear evaluation metric. The framework overhead is only justified when you can measure improvement quantitatively.

**7.** Static examples are fine when: inputs are homogeneous in domain and style, the task has one clear pattern, and you tested that 5 examples cover the distribution. Dynamic retrieval is justified when: the task spans multiple domains, some inputs are rare edge cases not covered by your static set, or testing shows systematic errors on specific input types.